In [0]:
%python
# Força o Spark a usar o fuso horário de Brasília para datas e horas
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

In [0]:
%python
# dml/01_carga_dim_fundo_imobiliario.ipynb
# MAGIC %pip install requests pandas

# %%
import requests
import zipfile
import io
import pandas as pd
from datetime import datetime
from pyspark.sql import functions as F

# Configurações de ambiente
catalogo = "product_dev"
schema = "financas"
tabela_dim = f"{catalogo}.{schema}.dim_fundo_imobiliario"

print("🚀 Iniciando Pipeline de Enriquecimento da Dimensão de FIIs...")
# 2. Carrega a sua base atual de FIIs (B3)
print("2. Carregando dados cadastrais atuais de FIIs...")
df_dim_atual = spark.sql(f"SELECT ticker, razao_social, nome_fundo, codigo_fundo, classificacao FROM {tabela_dim}")

# %%
# 3. Faz o download do arquivo ZIP de 2026 da CVM
url_zip = "https://dados.cvm.gov.br/dados/FII/DOC/INF_MENSAL/DADOS/inf_mensal_fii_2026.zip"
headers = {'User-Agent': 'Mozilla/5.0'}

print("3. Baixando arquivo consolidado de 2026 diretamente da CVM...")
response = requests.get(url_zip, headers=headers, timeout=30)
response.raise_for_status()
print("   ✅ Download concluído com sucesso!")

# %%
# 4. Descompacta e lê o arquivo cadastral de dentro do ZIP
print("4. Abrindo arquivo compactado e lendo o arquivo cadastral 'geral'...")
zip_file = zipfile.ZipFile(io.BytesIO(response.content))
arquivo_geral = "inf_mensal_fii_geral_2026.csv"

with zip_file.open(arquivo_geral) as f:
    df_pd_cvm = pd.read_csv(f, sep=';', encoding='ISO-8859-1', on_bad_lines='skip')

# Converte para Spark DataFrame tipando as colunas como String
df_spark_cvm_raw = spark.createDataFrame(df_pd_cvm.astype(str))

# %%
# 5. Limpa os dados da CVM e isola o de-para CNPJ -> Ticker
print("5. Tratando dados da CVM e traduzindo Codigos ISIN para Tickers...")
df_cvm_mapeado = (
    df_spark_cvm_raw
    # Filtra apenas ISINs válidos de FIIs
    .filter(
        F.col("Codigo_ISIN").startswith("BR") & 
        (~F.col("Codigo_ISIN").startswith("BR0000")) &
        (F.col("Codigo_ISIN") != "nan")
    )
    # Extrai o Ticker do ISIN (Ex: BRFVPQCTF015 -> FVPQ + 11 = FVPQ11)
    .withColumn("ticker_cvm", F.concat(F.substring(F.col("Codigo_ISIN"), 3, 4), F.lit("11")))
    # Garante o formato estrito de ticker B3 de varejo
    .filter(F.col("ticker_cvm").rlike(r"^[A-Z]{4}11$"))
    # Remove as máscaras de formatação do CNPJ
    .withColumn("cnpj_limpo", F.regexp_replace(F.col("CNPJ_Fundo_Classe"), r"[\./-]", ""))
    # Seleciona apenas o de-para cadastral
    .select(
        F.col("ticker_cvm").alias("ticker_cvm"),
        F.col("cnpj_limpo").alias("cnpj"),
        F.col("Codigo_ISIN").alias("codigo_isin"),
        F.col("Nome_Fundo_Classe").alias("nome_fundo_cvm"),
        F.col("Nome_Administrador").alias("administrador")
    )
    .distinct()
)

# %%
# 6. Executa o LEFT JOIN para enriquecer a sua tabela original
print("6. Cruzando bases de dados (B3 + CVM) via LEFT JOIN...")
df_dim_enriquecida = (
    df_dim_atual.alias("b3")
    .join(
        df_cvm_mapeado.alias("cvm"),
        F.col("b3.ticker") == F.col("cvm.ticker_cvm"),
        "left"
    )
    .select(
        F.col("b3.ticker").alias("ticker"),
        F.col("b3.razao_social").alias("razao_social"),
        F.col("b3.nome_fundo").alias("nome_fundo"),
        F.col("b3.codigo_fundo").alias("codigo_fundo"),
        F.col("b3.classificacao").alias("classificacao"),
        # Campos enriquecidos
        F.col("cvm.cnpj").alias("cnpj"),
        F.col("cvm.codigo_isin").alias("codigo_isin"),
        F.col("cvm.nome_fundo_cvm").alias("nome_fundo_cvm"),
        F.col("cvm.administrador").alias("administrador"),
        # Metadado de auditoria
        F.current_timestamp().alias("data_carga")
    )
)

# %%
# 7. Gravação física definitiva na tabela Delta original usando INSERT OVERWRITE
print(f"7. Gravando dados enriquecidos de forma atômica na tabela Delta '{tabela_dim}'...")

# Escreve os dados fazendo o overwrite de forma segura
df_dim_enriquecida.write.format("delta").mode("overwrite").saveAsTable(tabela_dim)

print("✅ PIPELINE CONCLUÍDO COM SUCESSO!")
print("   Sua dimensão de FIIs está física e definitivamente enriquecida com dados reais da CVM!")

# %%
# 8. Amostra para validação visual rápida
display(spark.sql(f"SELECT * FROM {tabela_dim} LIMIT 10"))